# core

> Utilities for managing SLURM jobs, SSH connections, and port forwarding on HPC clusters.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import test_eq, test_ne, test_fail

In [ ]:
#| export
import socket


In [ ]:
#| export
def find_free_port(above=8000):
    "Find a free port above `above`"
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('', 0))
        port = s.getsockname()[1]
        while port <= above:
            s.close()
            s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            s.bind(('', 0))
            port = s.getsockname()[1]
        return port

In [ ]:
port = find_free_port()
assert port > 8000, f"Expected port > 8000, got {port}"
assert port < 65536
print(f"Found free port: {port}")

In [ ]:
#| export
import subprocess


In [ ]:
#| export
def start_or_connect(job_name, host,
                     slurm_args="--qos=debug --gpus=1 --cpus-per-task=8 --mem=200G --time=01:00:00", return_cmd=False):
    """SSH to host, start a tmux session, and allocate a SLURM job inside it.

    Checks (via ssh+squeue) whether a SLURM job named `job_name` is already running.
    - If running: reattaches to the existing tmux session on the login node.
    - If not running: creates a tmux session that runs `salloc` to allocate resources.

    params:
        job_name: name for both the SLURM job and tmux session
        host: SSH host alias for the login node (must be in ~/.ssh/config)
        slurm_args: arguments passed to salloc (resources, time, qos, etc.)
        return_cmd: if True, return the command string; otherwise just print it
    """
    # Check if job is already running
    check = subprocess.run(
        ["ssh", host, f"squeue --me --name={job_name} --states=RUNNING --noheader"],
        capture_output=True, text=True
    )
    job_running = bool(check.stdout.strip())
    
    salloc_cmd = f"salloc --job-name={job_name} {slurm_args}"

    if job_running:
        print(f"Job '{job_name}' is already running. Reattaching to tmux...")
        ssh_cmd = f"ssh -t {host} \"tmux new-session -A -s {job_name}\""
    else:
        print(f"No running job '{job_name}' found. Starting salloc...")
        ssh_cmd = f"ssh -t {host} \"tmux new-session -A -s {job_name} '{salloc_cmd}'\""


    if return_cmd:
        return ssh_cmd
    else:
        print(f"Run this in your terminal:\n{ssh_cmd}")
        return

In [ ]:
from unittest.mock import patch, MagicMock


In [ ]:

def _mock_check(stdout=""):
    """Create a mock subprocess.run result with given stdout."""
    mock_result = MagicMock()
    mock_result.stdout = stdout
    mock_result.stderr = ""
    return mock_result

# Case 1: Job IS running → reattach (same salloc command either way)
with patch("subprocess.run", return_value=_mock_check("g3098")):
    cmd = start_or_connect("proxy_jump", "klone-login", slurm_args="--time=01:00:00", return_cmd=True)
test_eq(cmd, 'ssh -t klone-login "tmux new-session -A -s proxy_jump \'salloc --job-name=proxy_jump --time=01:00:00\'"')

print("---")

# Case 2: Job NOT running → same command, tmux -A handles both cases
with patch("subprocess.run", return_value=_mock_check("")):
    cmd = start_or_connect("proxy_jump", "klone-login", slurm_args="--time=01:00:00", return_cmd=True)
test_eq(cmd, 'ssh -t klone-login "tmux new-session -A -s proxy_jump \'salloc --job-name=proxy_jump --time=01:00:00\'"')

print("---")

# Case 3: return_cmd=False prints instead of returning
with patch("subprocess.run", return_value=_mock_check("g3098")):
    result = start_or_connect("proxy_jump", "klone-login", return_cmd=False)
test_eq(result, None)

In [ ]:
#| export
def job_stat(job_name, host,silent=False):
    """Get status info for a running SLURM job and print helpful commands.

    SSHs to `host` and queries squeue for the job's ID and node.
    Prints helper commands for getting a terminal, cancelling the job, etc.

    params:
        job_name: name of the SLURM job
        host: SSH host alias for the login node
    returns:
        dict with 'job_id' and 'node' keys, or None if no job found
    """
    result = subprocess.run(
        ["ssh", host,
         f"squeue --me --name={job_name} --states=RUNNING --Format=JobID,NodeList,TimeLeft,Partition --noheader"],
        capture_output=True, text=True
    )
    output = result.stdout.strip()
    if not output:
        if not silent:
            print(f"No running job '{job_name}' found on {host}.")
        return None

    parts = output.split()
    job_id = parts[0]
    node = parts[1]
    time_left = parts[2] if len(parts) > 2 else "?"
    partition = parts[3] if len(parts) > 3 else "?"

    if not silent:
        print(f"Job '{job_name}' status on {host}:")
        print(f"  Job ID:    {job_id}")
        print(f"  Node:      {node}")
        print(f"  Time left: {time_left}")
        print(f"  Partition: {partition}")
        print()
        print("Useful commands (run these on the login node or via ssh):")
        print(f"  # Get a shell on the compute node:")
        print(f"  ssh -t {host} 'srun --jobid={job_id} --overlap --pty bash'")
        print(f"  # Cancel the job:")
        print(f"  ssh {host} 'scancel {job_id}'")
        print(f"  # View job details:")
        print(f"  ssh {host} 'scontrol show job {job_id}'")

    return node, job_id

In [ ]:
# Test job_stat with a running job
with patch("subprocess.run", return_value=_mock_check("58903 g001 0:55:15 gpu-h200")):
    info = job_stat("proxy_jump", "klone-login")
test_eq(info, ("g001", "58903"))

print("---")

# Test job_stat with no running job
with patch("subprocess.run", return_value=_mock_check("")):
    info = job_stat("nope", "klone-login")
test_eq(info, None)

In [ ]:
#| export
def get_port_forwarding_command(local_port, remote_port, host, node, local_network="hyak.local"):
    """Get the command to forward local_port to remote_port on a compute node via the login host.
    Uses SSH local port forwarding with a ProxyJump through the login node.
    This function would not run the actual command, just print it to screen.
    """
    cmd = (
        f"ssh -N -f "
        f"-L {local_port}:{node}.{local_network}:{remote_port} "
        f"{host}"
    )
    print(cmd)
    return cmd

In [ ]:
cmd = get_port_forwarding_command(8080, 8000, "klone-login", "g3098")
test_eq(cmd, "ssh -N -f -L 8080:g3098.hyak.local:8000 klone-login")

# Custom network
cmd2 = get_port_forwarding_command(9090, 9000, "klone-login", "n1234", local_network="internal.net")
test_eq(cmd2, "ssh -N -f -L 9090:n1234.internal.net:9000 klone-login")

In [ ]:
#| export
import re
from pathlib import Path


In [ ]:
#| export
def update_ssh_node_config(job_name, host, config_path=None):
    """Query the SLURM node running `job_name` and update the SSH node config with its hostname.

    SSHs to `host` (login node) to find which compute node is running the job,
    then rewrites the `Hostname` line in the SSH sub-config file so that
    VSCode (or plain ssh) can ProxyJump directly to the compute node.

    params:
        job_name: name of the running SLURM job
        host: SSH host alias for the login node (e.g. 'klone-login')
        config_path: path to the SSH node config file. If None, defaults to
                     ~/.ssh/{cluster}-node-config where cluster is derived
                     from host (e.g. 'klone-login' → 'klone').
    returns:
        the node hostname string
    """
    # Get the node name 
    node,_ = job_stat(job_name, host,silent=True)
    if not node:
        raise RuntimeError(f"No running SLURM job named '{job_name}' found on {host}")

    # Determine config file path
    if config_path is None:
        cluster = host.replace("-login", "")
        config_path = Path.home() / ".ssh" / f"{cluster}-node-config"
    else:
        config_path = Path(config_path)

    # Read and update the Hostname line
    content = config_path.read_text()
    new_content = re.sub(r"(Hostname)\s+\S+", f"\\1 {node}", content)
    config_path.write_text(new_content)

    print(f"Updated {config_path}: Hostname → {node}")
    return node

In [ ]:
import tempfile, os


In [ ]:

# Test: updates Hostname line in a config file
sample_config = """Host klone-node
  User your-netid
  Hostname g3098
  ProxyJump klone-login
"""

with tempfile.NamedTemporaryFile(mode="w", suffix="-node-config", delete=False) as f:
    f.write(sample_config)
    tmp_path = f.name

try:
    with patch("subprocess.run", return_value=_mock_check("n1234")):
        node = update_ssh_node_config("proxy_jump", "klone-login", config_path=tmp_path)
    test_eq(node, "n1234")

    updated = Path(tmp_path).read_text()
    assert "Hostname n1234" in updated, f"Expected 'Hostname n1234' in:\n{updated}"
    assert "Hostname g3098" not in updated
    print(f"Updated config:\n{updated}")
finally:
    os.unlink(tmp_path)

# Test: raises when no job found
with patch("subprocess.run", return_value=_mock_check("")):
    test_fail(lambda: update_ssh_node_config("nope", "klone-login", config_path="/tmp/fake"),
              contains="No running SLURM job")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()